# Paper 2 — IUUY hybrid

Implements ≥80% top-two retrieval and explicit unmatched/ambiguous fallback. A fine-tuned/RLHF claim additionally requires hashed checkpoint, training-log, evaluation, deployment, and preference artifacts; this notebook does not invent them. For Colab, clone the repository and run `%pip install -e .[gemini,audio,diarization,evaluation]` first.


In [ ]:
from pathlib import Path
import os
import sys

search_roots = (Path.cwd(), *Path.cwd().parents, Path("/content/iedi-mas"))
ROOT = next((path for path in search_roots if (path / "src" / "iedi").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Repository not found. Clone it and install with: pip install -e .[gemini]")
sys.path.insert(0, str(ROOT / "src"))

from iedi.codebook import Codebook
from iedi.providers import GoogleGenAIProvider, OfflineFixtureProvider
from iedi.pipeline import build_pipeline
from iedi.schemas import InterpretationRequest

codebook = Codebook.from_json(ROOT / "data" / "codebook.demo.json")
# OfflineFixtureProvider only echoes reviewed evidence; it is never empirical evidence.
# Set IEDI_LIVE_GEMINI=1 and GEMINI_API_KEY to exercise the real 2.5 Flash/Pro adapter.
LIVE_GEMINI = os.getenv("IEDI_LIVE_GEMINI") == "1"
provider = GoogleGenAIProvider() if LIVE_GEMINI else OfflineFixtureProvider()
print("provider:", "live Gemini" if LIVE_GEMINI else "offline schema fixture")

pipeline = build_pipeline("paper2", codebook=codebook, provider=provider, config_path=ROOT / "configs" / "paper2.json")


In [ ]:
request = InterpretationRequest(
    utterance="How far?",
    speaker_id="SPEAKER_00",
)
result = pipeline.interpret(request)
result.to_dict()


For audio, construct `InputAgent` with `WhisperASR` and `PyannoteDiarizer`, using `require_diarization=True`. Missing diarization then fails visibly instead of returning a constant speaker label.
